## Implement a shallow auto encoder and decoder network for machine translation (by using Kaggle English to Hindi neural translation dataset).

You can download the dataset from the below link

https://www.kaggle.com/datasets/vaibhavkumar11/hindi-english-parallel-corpus/data

##### Import Necessary Libraries

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split



#### Loading and Sampling the Dataset
Next, we'll load the Kaggle English to Hindi translation dataset and sample a smaller subset for training.

In [ ]:
# Load the dataset
data_path = 'hindi_english_parallel.csv'  # Ensure the dataset is in the working directory
df = pd.read_csv(data_path)

# Sample the data to reduce memory usage
df_sampled = df.sample(n=10000, random_state=42)  # Adjust the sample size as needed

# Ensure there are no null values in the sampled data
df_sampled = df_sampled.dropna(subset=['hindi', 'english'])

# Select specific rows to display consistently
display_rows = df_sampled.iloc[:5]

# Display the selected rows
print("Selected sample rows:")
print(display_rows)


Selected sample rows:
                                                     hindi  \
957248   बडे पैमाने पर सुनामी से प्रभावीत जापान में 4 द...   
1072034                           वर्ग का पूर्णा क्या था?    
1195844                         मैं अपना काम कर चुका हूँ।    
1123517                  राष्ट्रीय मनः स्वास्थ्य कार्यक्रम   
933515                                           क्रियावली   

                                                   english  
957248   4 days after the massive tsunami struck Japan,...  
1072034                    What was completing the square?  
1195844                       I have already done my work.  
1123517                   National Mental Health Programme  
933515                                                menu  


##### Preprocessing DataWe need to preprocess the data by cleaning, tokenizing, and padding the sequences.

- Cleaning: Convert text to lowercase and strip any leading or trailing spaces.

- Tokenizing: Convert words into numeric tokens.

- Padding: Ensure all sequences are the same length by padding shorter sequences with zeros.

In [ ]:
# Clean the text data
def clean_text(text):
    text = text.lower().strip()
    return text

df_sampled['english'] = df_sampled['english'].apply(clean_text)
df_sampled['hindi'] = df_sampled['hindi'].apply(clean_text)

# Print cleaned data samples
print("Cleaned English text samples:")
print(df_sampled['english'].iloc[:5])
print("\nCleaned Hindi text samples:")
print(df_sampled['hindi'].iloc[:5])


Cleaned English text samples:
957248     4 days after the massive tsunami struck japan,...
1072034                      what was completing the square?
1195844                         i have already done my work.
1123517                     national mental health programme
933515                                                  menu
Name: english, dtype: object

Cleaned Hindi text samples:
957248     बडे पैमाने पर सुनामी से प्रभावीत जापान में 4 द...
1072034                              वर्ग का पूर्णा क्या था?
1195844                            मैं अपना काम कर चुका हूँ।
1123517                    राष्ट्रीय मनः स्वास्थ्य कार्यक्रम
933515                                             क्रियावली
Name: hindi, dtype: object


In [ ]:
# Tokenize the text data
eng_tokenizer = Tokenizer()
hin_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(df_sampled['english'])
hin_tokenizer.fit_on_texts(df_sampled['hindi'])

# Print the word index for English and Hindi
print("\nEnglish word index:")
print(list(eng_tokenizer.word_index.items())[:10])
print("\nHindi word index:")
print(list(hin_tokenizer.word_index.items())[:10])



English word index:
[('the', 1), ('of', 2), ('and', 3), ('to', 4), ('in', 5), ('a', 6), ('is', 7), ('for', 8), ('that', 9), ('you', 10)]

Hindi word index:
[('के', 1), ('और', 2), ('में', 3), ('है', 4), ('की', 5), ('से', 6), ('को', 7), ('का', 8), ('कि', 9), ('है।', 10)]


In [ ]:
# Convert text to sequences
eng_sequences = eng_tokenizer.texts_to_sequences(df_sampled['english'])
hin_sequences = hin_tokenizer.texts_to_sequences(df_sampled['hindi'])

# Print sequences for the first few English and Hindi samples
print("\nEnglish sequences:")
print(eng_sequences[:5])
print("\nHindi sequences:")
print(hin_sequences[:5])



English sequences:
[[395, 295, 76, 1, 5051, 7572, 2635, 2256, 2257, 2, 2258, 1181, 278, 2259, 55, 7573], [51, 26, 7574, 1, 1079], [36, 22, 259, 330, 78, 146], [130, 1289, 432, 664], [1734]]

Hindi sequences:
[[3291, 2154, 11, 8190, 6, 8191, 2427, 3, 606, 8192, 79, 34, 335, 56, 5381, 93, 5, 8193, 8194, 19, 181, 182], [607, 8, 8195, 47, 42], [63, 150, 108, 23, 456, 668], [183, 8196, 551, 532], [8197]]


In [ ]:
# Pad sequences to the same length
max_eng_len = 20  # Limiting max length to 20 for practical purposes
max_hin_len = 20

eng_padded = pad_sequences(eng_sequences, maxlen=max_eng_len, padding='post')
hin_padded = pad_sequences(hin_sequences, maxlen=max_hin_len, padding='post')

# Print the padded sequences for the first few English and Hindi samples
print("\nPadded English sequences:")
print(eng_padded[:5])
print("\nPadded Hindi sequences:")
print(hin_padded[:5])



Padded English sequences:
[[ 395  295   76    1 5051 7572 2635 2256 2257    2 2258 1181  278 2259
    55 7573    0    0    0    0]
 [  51   26 7574    1 1079    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [  36   22  259  330   78  146    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [ 130 1289  432  664    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [1734    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]]

Padded Hindi sequences:
[[  11 8190    6 8191 2427    3  606 8192   79   34  335   56 5381   93
     5 8193 8194   19  181  182]
 [ 607    8 8195   47   42    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [  63  150  108   23  456  668    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [ 183 8196  551  532    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [

##### Train-Test Split
We split the data into training and testing sets to evaluate the model's performance on unseen data.

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(eng_padded, hin_padded, test_size=0.2, random_state=42)


##### Building the Model
We'll build a simple autoencoder-decoder model using LSTM layers.

- Encoder: Processes the input sequence and compresses it into a context vector.

- Decoder: Generates the output sequence from the context vector.

In [ ]:
  # Define model parameters
  latent_dim = 128  # Reduced latent dimensionality for CPU-friendly processing
  num_eng_tokens = len(eng_tokenizer.word_index) + 1
  num_hin_tokens = len(hin_tokenizer.word_index) + 1

  # Encoder
  encoder_inputs = Input(shape=(None,))
  enc_emb = Embedding(num_eng_tokens, latent_dim, mask_zero=True)(encoder_inputs)
  encoder_lstm = LSTM(latent_dim, return_state=True)
  encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
  encoder_states = [state_h, state_c]

  # Decoder
  decoder_inputs = Input(shape=(None,))
  dec_emb = Embedding(num_hin_tokens, latent_dim, mask_zero=True)(decoder_inputs)
  decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
  decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
  decoder_dense = Dense(num_hin_tokens, activation='softmax')
  decoder_outputs = decoder_dense(decoder_outputs)

  # Model
  model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

  # Summarize the model
  model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, None)]       0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, None)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, None, 128)    2198912     ['input_1[0][0]']                
                                                                                                  
 embedding_1 (Embedding)        (None, None, 128)    2668160     ['input_2[0][0]']                
                                                                                              

##### Compiling the Model
We'll compile the model using an optimizer and a loss function suitable for multi-class classification.

In [ ]:
# Compile the model
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')


##### Training the Model
We'll train the model with the training data.

- Epochs: Number of times the model will see the entire training dataset.

- Batch Size: Number of samples processed before the model is updated.

In [ ]:
# Reshape y_train for sparse_categorical_crossentropy
y_train = y_train.reshape(y_train.shape[0], y_train.shape[1], 1)

# Train the model
history = model.fit([X_train, y_train[:, :-1]], y_train[:, 1:],
                    batch_size=32,  # Smaller batch size for CPU-friendly training
                    epochs=5,  # Reduced epochs for practical training time
                    validation_split=0.2)


Epoch 1/5
200/200 [==============================] - 24s 99ms/step - loss: 3.8669 - val_loss: 3.5861
Epoch 2/5
200/200 [==============================] - 22s 112ms/step - loss: 3.5597 - val_loss: 3.5414
Epoch 3/5
200/200 [==============================] - 21s 103ms/step - loss: 3.4775 - val_loss: 3.5067
Epoch 4/5
200/200 [==============================] - 23s 113ms/step - loss: 3.4060 - val_loss: 3.4727
Epoch 5/5
200/200 [==============================] - 22s 111ms/step - loss: 3.3406 - val_loss: 3.4517


##### Evaluating the Model
We'll evaluate the model's performance on the test set to see how well it can translate unseen data.

In [ ]:
# Reshape y_test for sparse_categorical_crossentropy
y_test = y_test.reshape(y_test.shape[0], y_test.shape[1], 1)

# Evaluate the model on the test set
loss = model.evaluate([X_test, y_test[:, :-1]], y_test[:, 1:])
print(f'Test Loss: {loss}')

63/63 [==============================] - 2s 36ms/step - loss: 3.5165
Test Loss: 3.516465425491333


##### Inference Setup
Setting up the encoder and decoder for inference to use the trained model for translations.

In [ ]:
# Define the encoder model for inference
encoder_model = Model(encoder_inputs, encoder_states)

# Define the decoder model for inference
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h, state_c = decoder_lstm(dec_emb, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states)


##### Prediction Function
Creating a function to translate English sentences to Hindi using the trained model.

In [ ]:
def decode_sequence(input_seq):
    # Encode the input as state vectors
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1 with start token
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = hin_tokenizer.word_index.get('starttoken', 0)

    # Loop to generate the translation
    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        # Predict the next word in the sequence
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Get the most likely next word
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = hin_tokenizer.index_word.get(sampled_token_index, '')

        # Append the word to the translated sentence
        decoded_sentence += ' ' + sampled_word

        # Exit condition: hit max length or end token
        if sampled_word == 'endtoken' or len(decoded_sentence.split()) > max_hin_len:
            stop_condition = True

        # Update the target sequence (of length 1)
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return decoded_sentence


#### Example Sentences for Testing
We’ll use a few example English sentences and see how they are translated into Hindi.

In [ ]:
# Example sentences for testing
test_sentences = [
    "The color and opacity of the highlight fill.",
    "This is a simple example sentence.",
    "The model will translate these sentences."
]

# Preprocess the test sentences
test_sequences = eng_tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_sequences, maxlen=max_eng_len, padding='post')

# Translate the sentences
for seq in test_padded:
    input_seq = np.array([seq])
    translated_sentence = decode_sequence(input_seq)
    print("English Sentence:", " ".join([eng_tokenizer.index_word.get(i, '') for i in seq if i != 0]))
    print("Translated Hindi Sentence:", translated_sentence)
    print("\n")


1/1 [==============================] - 0s 23ms/step
English Sentence: the color and of the highlight fill
Translated Hindi Sentence:  लिए के लिए नहीं  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए


1/1 [==============================] - 0s 23ms/step
English Sentence: this is a simple example sentence
Translated Hindi Sentence:  लिए के लिए के लिए नहीं  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए


1/1 [==============================] - 0s 24ms/step
English Sentence: the model will translate these
Translated Hindi Sentence:  लिए के लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए  लिए


